# Notebook 02: Implied-Volatility Surface Fitting

This notebook takes the cleaned option data from Notebook 1 and turns it into a smooth cross-section of market-implied volatility. The logic is sequential: invert prices to implied vols, interpolate total variance across strike and maturity, convert that surface into call prices, and then enforce static no-arbitrage on the final exported surface.


### Why We Work in Implied Volatility Instead of Raw Prices

This notebook takes the cleaned option data from Notebook 1 and turns it into a smooth cross-section of market-implied volatility. In the production pipeline, the repo first merges OTM puts and OTM calls into a wider synthetic call surface using put-call parity. In this notebook we keep the example simpler, but the modeling logic is the same: invert prices to implied vols, organize the data in log-moneyness and maturity, interpolate total variance, convert that surface into call prices, and then enforce static no-arbitrage on the exported grid.


In [ ]:
# Cell 0: Setup Python Path
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add src directory to path so modules are importable at top level
src_path = Path.cwd().parent / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f"Added to path: {src_path}")


In [ ]:
# Cell 1: Load cleaned data from Notebook 1
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Create output directory
output_dir = Path("../output")
output_dir.mkdir(parents=True, exist_ok=True)

# Load the cleaned data saved by notebook 1 (canonical final dataset)
df = pd.read_csv("../output/cleaned_options_data.csv")


# Extract market parameters
S0 = float(df['underlying_price'].iloc[0])
r = float(df['risk_free_rate'].iloc[0])
q = float(df['dividend_yield'].iloc[0])

print(f"Loaded {len(df)} cleaned options")
print(f"Market parameters: S0={S0:.2f}, r={r:.4f}, q={q:.4f}")
print(f"Maturity range: T ∈ [{df['time_to_expiry'].min():.4f}, {df['time_to_expiry'].max():.4f}]")
print(df.head())



At this point we have cleaned prices, but not yet a model surface. The next steps compute implied vol for each quote, convert it into total variance, and organize the data into maturity slices. Those slices are then interpolated across strike and maturity with a shape-preserving total-variance surface.


In [ ]:
# Cell 2: Compute IV, log-moneyness, and total variance
from implied_volatility.iv_solver import implied_vol, IVSolverConfig

# Compute implied volatility for each option
iv_cfg = IVSolverConfig(method="hybrid", max_iter=100, tol=1e-10)

df['iv'] = implied_vol(
    prices=df['mid'].values,
    S=df['underlying_price'].values,
    K=df['strike'].values,
    T=df['time_to_expiry'].values,
    r=r,
    q=q,
    option_type='call',
    cfg=iv_cfg,
)

n_before = len(df)
df = df[
    (df['iv'] > 0.05) &      # Min 5% vol
    (df['iv'] < 1.50) &      # Max 150% vol (was seeing higher?)
    (~df['iv'].isna())       # No NaN
].copy()
n_after = len(df)

print(f"📊 IV Quality Filter:")
print(f"   Before: {n_before} options")
print(f"   After: {n_after} options ({100*n_after/n_before:.1f}%)")

# Compute log-moneyness and total variance
df['log_moneyness'] = np.log(df['strike'] / S0)
df['w'] = (df['iv'] ** 2) * df['time_to_expiry']

print(f"Computed IV and derived columns:")
print(f"  IV range: [{df['iv'].min():.4f}, {df['iv'].max():.4f}]")
print(f"  log_moneyness range: [{df['log_moneyness'].min():.3f}, {df['log_moneyness'].max():.3f}]")
print(f"  w range: [{df['w'].min():.6f}, {df['w'].max():.6f}]")



In [ ]:
# Cell 3: Group data by maturity into slices for surface fitting

grouped = (
    df.groupby("time_to_expiry")
      .filter(lambda g: len(g) >= 8)   # ← critical
      .groupby("time_to_expiry")
)

T_slices = []
x_slices = []
w_slices = []

MIN_POINTS_PER_SLICE = 8

for T, group in grouped:
    group_sorted = group.sort_values('log_moneyness')

    # Skip slices with too few points
    if len(group_sorted) < MIN_POINTS_PER_SLICE:
        print(f"⚠️ Skipping T={T:.4f}: only {len(group_sorted)} points (min {MIN_POINTS_PER_SLICE})")
        continue
    
    T_slices.append(float(T))
    x_slices.append(group_sorted['log_moneyness'].values)
    w_slices.append(group_sorted['w'].values)

print(f"Prepared {len(T_slices)} maturity slices:")
for i, T in enumerate(T_slices[:5]):
    print(f"  T={T:.4f}: {len(x_slices[i])} points, x ∈ [{x_slices[i].min():.3f}, {x_slices[i].max():.3f}]")



The repo's canonical surface builder is now a direct interpolated total-variance surface rather than a parametric slice model. That is more robust on noisy live option snapshots. We still treat the raw interpolated surface as an intermediate object: it is useful for smoothing the observed quotes, but the final exported surface is only trusted after the call grid has been projected to satisfy static no-arbitrage.


In [ ]:
# Cell 4: Fit the interpolated total-variance surface
from surface_fitting.spline import fit_spline_surface

x_mins = [float(np.min(x)) for x in x_slices]
x_maxs = [float(np.max(x)) for x in x_slices]
x_lo = max(x_mins)
x_hi = min(x_maxs)
if not np.isfinite(x_lo) or not np.isfinite(x_hi) or x_hi <= x_lo:
    x_lo = max(float(np.quantile(x, 0.05)) for x in x_slices)
    x_hi = min(float(np.quantile(x, 0.95)) for x in x_slices)
if not np.isfinite(x_lo) or not np.isfinite(x_hi) or x_hi <= x_lo:
    x_all = np.concatenate(x_slices)
    x_lo = float(np.min(x_all))
    x_hi = float(np.max(x_all))

x_grid_common = np.linspace(x_lo, x_hi, 61)
T_grid_surface = np.sort(np.asarray(T_slices, dtype=float))

variance_surface = fit_spline_surface(
    x_slices=x_slices,
    w_slices=w_slices,
    T_slices=T_slices,
    x_grid=x_grid_common,
    T_grid=T_grid_surface,
    kind_x='pchip',
    kind_T='pchip',
    method_2d='linear',
)

print('Interpolated total-variance surface fitted successfully')
print(f'  x_grid range: [{variance_surface.x_grid.min():.3f}, {variance_surface.x_grid.max():.3f}]')
print(f'  T_grid range: [{variance_surface.T_grid.min():.4f}, {variance_surface.T_grid.max():.4f}]')
print(f'  w_grid shape: {variance_surface.w_grid.shape}')


In [ ]:
# Cell 5: Summarize the maturity slices feeding the surface
surface_slice_summary = pd.DataFrame([
    {
        'T': float(T),
        'n_points': int(len(x)),
        'x_min': float(np.min(x)),
        'x_max': float(np.max(x)),
        'w_min': float(np.min(w)),
        'w_max': float(np.max(w)),
    }
    for T, x, w in zip(T_slices, x_slices, w_slices)
]).sort_values('T').reset_index(drop=True)

print(surface_slice_summary.to_string(index=False))


In [ ]:
# Cell 6: Visualize slice coverage across maturity
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(surface_slice_summary['T'], surface_slice_summary['n_points'], 'o-', lw=2)
axes[0].set_title('Available Quotes per Maturity', fontsize=12)
axes[0].set_xlabel('Maturity T (years)', fontsize=10)
axes[0].set_ylabel('Number of quotes', fontsize=10)
axes[0].grid(True, alpha=0.3)

axes[1].fill_between(
    surface_slice_summary['T'],
    surface_slice_summary['x_min'],
    surface_slice_summary['x_max'],
    alpha=0.30,
    color='steelblue',
    label='observed x-range',
)
axes[1].plot(surface_slice_summary['T'], surface_slice_summary['x_min'], color='steelblue', lw=1.5)
axes[1].plot(surface_slice_summary['T'], surface_slice_summary['x_max'], color='steelblue', lw=1.5)
axes[1].axhline(variance_surface.x_grid.min(), color='darkorange', linestyle='--', lw=1.5, label='surface x-grid min/max')
axes[1].axhline(variance_surface.x_grid.max(), color='darkorange', linestyle='--', lw=1.5)
axes[1].set_title('Observed Log-Moneyness Coverage', fontsize=12)
axes[1].set_xlabel('Maturity T (years)', fontsize=10)
axes[1].set_ylabel('Log-moneyness x', fontsize=10)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../output/total_variance_slice_coverage.png', dpi=150, bbox_inches='tight')
plt.show()


These plots show whether the interpolation is being asked to bridge modest gaps or large unsupported regions. A robust surface should have reasonable quote counts per expiry and a common interior log-moneyness range where most maturities overlap.


In [ ]:
# Cell 7: Check fit quality on a representative maturity (~30 days)
target_T = 30.0 / 365.0
T_grid = variance_surface.T_grid
idx = np.argmin(np.abs(T_grid - target_T))
T_sample = T_grid[idx]

x_sample = x_slices[idx]
w_sample = w_slices[idx]

x_fine = np.linspace(x_sample.min(), x_sample.max(), 200)
T_fine = np.full_like(x_fine, T_sample)
w_fit = variance_surface.total_variance(x_fine, T_fine)

plt.figure(figsize=(10, 6))
plt.plot(x_sample, w_sample, 'o', label='Market (total variance)', ms=7, alpha=0.7)
plt.plot(x_fine, w_fit, '-', label='Interpolated fit', lw=2.5)
plt.xlabel('Log-moneyness x', fontsize=11)
plt.ylabel('Total variance w = ??T', fontsize=11)
plt.title(f'Interpolated Total-Variance Fit at T ? {T_sample:.4f} years ({T_sample*365:.1f} days)', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'../output/total_variance_fit_slice_T{T_sample:.4f}.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Cell 8: Visualize the full interpolated total-variance surface
TT_surface, XX_surface = np.meshgrid(variance_surface.T_grid, variance_surface.x_grid, indexing='ij')

plt.figure(figsize=(11, 6))
plt.contourf(TT_surface, XX_surface, variance_surface.w_grid, levels=30, cmap='viridis')
plt.colorbar(label='Total variance w')
plt.xlabel('Maturity T (years)', fontsize=11)
plt.ylabel('Log-moneyness x', fontsize=11)
plt.title('Interpolated Total-Variance Surface', fontsize=12)
plt.tight_layout()
plt.savefig('../output/total_variance_surface.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Cell 9: Compare nearby slices against the interpolated surface
sample_indices = np.unique(np.clip([max(idx - 1, 0), idx, min(idx + 1, len(T_slices) - 1)], 0, len(T_slices) - 1))
plt.figure(figsize=(10, 6))
for sample_idx in sample_indices:
    T_here = T_slices[sample_idx]
    x_here = x_slices[sample_idx]
    w_here = w_slices[sample_idx]
    x_line = np.linspace(x_here.min(), x_here.max(), 200)
    w_line = variance_surface.total_variance(x_line, np.full_like(x_line, T_here))
    plt.plot(x_here, w_here, 'o', ms=5, alpha=0.55, label=f'Market T={T_here:.4f}')
    plt.plot(x_line, w_line, '-', lw=2.0, label=f'Fit T={T_here:.4f}')

plt.xlabel('Log-moneyness x', fontsize=11)
plt.ylabel('Total variance w', fontsize=11)
plt.title('Raw Slice Data Versus Interpolated Surface', fontsize=12)
plt.legend(fontsize=9, ncol=2)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../output/total_variance_neighbor_slices.png', dpi=150, bbox_inches='tight')
plt.show()


The raw interpolated surface is only an intermediate modeling object. The repo does not trust it blindly. Instead, it converts that surface into a call-price grid, checks the static-arbitrage conditions explicitly, and then projects the grid onto the no-arbitrage set before exporting anything downstream.


In [ ]:
# Cell 10: Convert the interpolated surface to a raw call-price grid
from implied_volatility.surface import call_price_grid_from_iv_surface

K_grid = np.linspace(df['strike'].min(), df['strike'].max(), 80)
T_grid = np.sort(df['time_to_expiry'].unique())

C_grid_raw = call_price_grid_from_iv_surface(
    iv_surface=variance_surface,
    S0=S0,
    K_grid=K_grid,
    T_grid=T_grid,
    r=r,
    q=q,
    coord='x_T',
)

print(f'Call price grid shape: {C_grid_raw.shape} (nT={len(T_grid)}, nK={len(K_grid)})')
print(f'Price range: [{C_grid_raw.min():.4f}, {C_grid_raw.max():.4f}]')


In [ ]:
# Cell 11: Run static-arbitrage checks and project the grid
from arbitrage.projection import project_call_surface_static_arbitrage
from arbitrage.static_checks import check_static_arbitrage_calls

arb_report_raw = check_static_arbitrage_calls(
    C=C_grid_raw,
    K=K_grid,
    T=T_grid,
    S0=S0,
    r=r,
    q=q,
    tol=1e-10,
    return_derivatives=True,
)

projection = project_call_surface_static_arbitrage(
    C=C_grid_raw,
    K=K_grid,
    T=T_grid,
    S0=S0,
    r=r,
    q=q,
    tol=1e-10,
)
C_grid_proj = projection.C_projected

arb_report_proj = check_static_arbitrage_calls(
    C=C_grid_proj,
    K=K_grid,
    T=T_grid,
    S0=S0,
    r=r,
    q=q,
    tol=1e-10,
    return_derivatives=True,
)

print('=' * 70)
print('STATIC ARBITRAGE CHECK - RAW INTERPOLATED SURFACE')
print('=' * 70)
for key, val in arb_report_raw.counts.items():
    print(f'  {key:30s}: {val:6d}')

print('
' + '=' * 70)
print('STATIC ARBITRAGE CHECK - PROJECTED SURFACE')
print('=' * 70)
for key, val in arb_report_proj.counts.items():
    print(f'  {key:30s}: {val:6d}')

pass_rate_raw = 100 * (1 - arb_report_raw.counts['n_fail'] / arb_report_raw.counts['n_total']) if arb_report_raw.counts['n_total'] > 0 else 100.0
pass_rate_proj = 100 * (1 - arb_report_proj.counts['n_fail'] / arb_report_proj.counts['n_total']) if arb_report_proj.counts['n_total'] > 0 else 100.0
print(f'
Raw pass rate:       {pass_rate_raw:6.2f}%')
print(f'Projected pass rate: {pass_rate_proj:6.2f}%')
print(f'Projection iterations: {projection.iterations}')


In [ ]:
# Cell 12: Plot arbitrage diagnostics before and after projection
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
TT, KK = np.meshgrid(T_grid, K_grid, indexing='ij')

axes[0,0].contourf(TT, KK, arb_report_raw.pass_mask.astype(float), levels=[0, 0.5, 1], colors=['red', 'green'], alpha=0.6)
axes[0,0].set_title('Raw Surface: Pass / Fail', fontsize=11)
axes[0,0].set_xlabel('Maturity T (years)', fontsize=10)
axes[0,0].set_ylabel('Strike K ($)', fontsize=10)

axes[0,1].contourf(TT, KK, arb_report_proj.pass_mask.astype(float), levels=[0, 0.5, 1], colors=['red', 'green'], alpha=0.6)
axes[0,1].set_title('Projected Surface: Pass / Fail', fontsize=11)
axes[0,1].set_xlabel('Maturity T (years)', fontsize=10)
axes[0,1].set_ylabel('Strike K ($)', fontsize=10)

im2 = axes[1,0].contourf(TT, KK, (~arb_report_raw.calendar_monotone_ok).astype(float), levels=10, cmap='Reds')
axes[1,0].set_title('Raw Surface: Calendar Violations', fontsize=11)
axes[1,0].set_xlabel('Maturity T (years)', fontsize=10)
axes[1,0].set_ylabel('Strike K ($)', fontsize=10)
plt.colorbar(im2, ax=axes[1,0])

im3 = axes[1,1].contourf(TT, KK, (~arb_report_raw.strike_convex_ok).astype(float), levels=10, cmap='Reds')
axes[1,1].set_title('Raw Surface: Convexity Violations', fontsize=11)
axes[1,1].set_xlabel('Maturity T (years)', fontsize=10)
axes[1,1].set_ylabel('Strike K ($)', fontsize=10)
plt.colorbar(im3, ax=axes[1,1])

plt.suptitle('Static Arbitrage Analysis: Raw Surface vs Projected Surface', fontsize=13, y=0.995)
plt.tight_layout()
plt.savefig('../output/arbitrage_projection_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()


The density step is a reality check. If the fitted call surface implies negative risk-neutral density, the surface is not suitable for Dupire. In this repo, the final canonical surface is the arbitrage-repaired call grid, not the unadjusted raw fit.


In [ ]:
# Cell 13: Extract risk-neutral density from the projected call-price surface
from arbitrage.density import breeden_litzenberger_density

density_report = breeden_litzenberger_density(
    C=C_grid_proj,
    K=K_grid,
    T=T_grid,
    r=r,
    tol=1e-12,
)

print('='*70)
print('BREEDEN-LITZENBERGER DENSITY EXTRACTION')
print('='*70)
for key, val in density_report.counts.items():
    print(f'  {key:20s}: {val:6d}')

print('
  Mass per maturity (first 10 slices, should ? 1.0):')
for i, T in enumerate(T_grid[:10]):
    print(f'    T={T:.4f}: mass={density_report.mass[i]:.6f}')
print('='*70)


In [ ]:
# Cell 14: Plot density for the representative maturity
idx_density = np.argmin(np.abs(T_grid - T_sample))
T_density = T_grid[idx_density]
p_density = density_report.density[idx_density, :]

plt.figure(figsize=(10, 6))
plt.plot(K_grid, p_density, '-', lw=2.5, color='navy')
plt.axhline(0, color='black', linestyle='--', alpha=0.3)
plt.fill_between(K_grid, 0, p_density, where=(p_density >= 0), alpha=0.3, color='green', label='Positive density')
plt.fill_between(K_grid, 0, p_density, where=(p_density < 0), color='red', alpha=0.4, label='Negative density (arbitrage)')
plt.xlabel('Strike K ($)', fontsize=11)
plt.ylabel('Risk-neutral density p(K,T)', fontsize=11)
plt.title(f'Breeden-Litzenberger Density at T ? {T_density:.4f} years ({T_density*365:.1f} days)', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'../output/density_T{T_density:.4f}.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Cell 15: Check density normalization (diagnostic)
from arbitrage.density import density_normalization_diagnostic

mass, ok_mask = density_normalization_diagnostic(
    density=density_report.density,
    K=K_grid,
    tol=1e-2,
)

print('
Density Mass Check (|mass - 1| <= 0.01):')
print(f'  Pass: {ok_mask.sum()} / {len(ok_mask)} maturities')
print(f'  Mass range: [{mass.min():.6f}, {mass.max():.6f}]')
print(f'  Mean mass: {mass.mean():.6f}')
print(f'  Std mass:  {mass.std():.6f}')


In [ ]:
# Cell 16: Save all outputs to disk
surface_slice_summary.to_csv('../output/surface_slice_summary.csv', index=False)

np.savez(
    '../output/call_price_grid.npz',
    C=C_grid_proj,
    C_raw=C_grid_raw,
    K=K_grid,
    T=T_grid,
    S0=S0,
    r=r,
    q=q,
)

np.savez(
    '../output/interpolated_total_variance_surface.npz',
    x_grid=variance_surface.x_grid,
    T_grid=variance_surface.T_grid,
    w_grid=variance_surface.w_grid,
)

np.savez(
    '../output/density.npz',
    density=density_report.density,
    mass=density_report.mass,
    K=K_grid,
    T=T_grid,
)

print('All results saved to ../output/')


The saved outputs from this notebook are the handoff to the local-vol stage. They include the raw fit diagnostics, the projected arbitrage-free call grid, and the arbitrage-free implied-volatility surface artifact that is backed by that projected grid.


In [ ]:
# Cell 17: Display comprehensive summary
print('
' + '='*70)
print('NOTEBOOK 02 SUMMARY: IV SURFACE FITTING')
print('='*70)
print(f'Total options processed:           {len(df)}')
print(f'Valid options (finite IV):         {len(df[np.isfinite(df["iv"])])}')
print(f'Number of maturity slices:         {len(T_slices)}')
print(f'Maturity range:                    [{min(T_slices):.4f}, {max(T_slices):.4f}] years')
print('
Interpolated total-variance surface: ?')
print('Projected arbitrage-free call grid: ?')
print('
Static Arbitrage Analysis:')
print(f'  Raw grid violations:             {arb_report_raw.counts["n_fail"]}')
print(f'  Projected grid violations:       {arb_report_proj.counts["n_fail"]}')
print(f'  Raw pass rate:                   {pass_rate_raw:.2f}%')
print(f'  Projected pass rate:             {pass_rate_proj:.2f}%')
print('
Breeden-Litzenberger Density:')
print(f'  Total points:                    {density_report.counts["n_total"]}')
print(f'  Negative density points:         {density_report.counts["n_negative"]}')
print(f'  Normalization pass rate:         {ok_mask.sum()}/{len(ok_mask)} maturities')
print('='*70)
print('
Notebook 02 complete. Proceed to Notebook 03 for Dupire LV computation.')
